In [1]:
%load_ext autoreload
%autoreload 2

import tiktoken
import torch
from utils import plot_metrics,  tokenize, group
import os

num_cores = os.cpu_count()
enc = tiktoken.get_encoding("gpt2")
enc.n_vocab

50257

In [2]:
from model import ModelConfig

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device} with {num_cores} cores")

config = ModelConfig()

Using device: mps with 8 cores


In [3]:
import datasets

ds = datasets.load_dataset('roneneldan/TinyStories', streaming=True)
demo_samples = ds['train'].take(3)

for i, example in enumerate(demo_samples):
    print(f"--- Story {i+1} ---")
    print(example['text'])
    

--- Story 1 ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
--- Story 2 ---
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had 

In [4]:
dataset = ds.map(
    tokenize, 
    batched=True, 
    remove_columns=["text"]
).map(
    group, 
    batched=True, 
    fn_kwargs={"block_size": config.context_length}, 
).with_format('torch')

example = next(iter(dataset['train']))
example['encoded'].shape, enc.decode(example['encoded'].numpy())

(torch.Size([9]), 'One day, a little girl named Lily found')

In [5]:
from torch.utils.data import DataLoader

train_dataset = dataset["train"].shuffle(buffer_size=10000)
val_dataset = dataset["validation"]

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, pin_memory=True, num_workers=min(4, num_cores))  # type: ignore
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, pin_memory=True, num_workers=min(2, num_cores))  # type: ignore

In [6]:
from torch import optim, nn
import torch.nn.functional as F
import torchmetrics
from model import BigramLanguageModel, Sampler, Transformer

model = Transformer(config).to(device)
optimizer = optim.AdamW(model.parameters(), config.learning_rate)
metrics = torchmetrics.MetricCollection({
    'loss': torchmetrics.MeanMetric()
})

def compute_loss(logits: torch.Tensor, targets: torch.Tensor):
    B, T, C = logits.shape
    return F.cross_entropy(logits.reshape(B * T, C), targets.reshape(B * T))

def train_step(model: nn.Module, batch, optimizer: optim.Optimizer, metrics: torchmetrics.MetricCollection):
    data = batch["encoded"].to(device)
    x = data[:, :-1]
    y = data[:, 1:]

    optimizer.zero_grad(set_to_none=True)

    logits = model(x)  
    loss = compute_loss(logits, y)

    loss.backward()
    optimizer.step()
    metrics.update(value=loss.item())

@torch.no_grad()
def eval_step(model: nn.Module, batch, metrics: torchmetrics.MetricCollection):
    data = batch["encoded"].to(device)
    x = data[:, :-1]
    y = data[:, 1:]
    logits = model(x)
    loss = compute_loss(logits, y)
    metrics.update(value=loss.item())


In [7]:
x, y = example['encoded'][:-1].to(device), example['encoded'][1:].to(device)
logits = model(x.unsqueeze(0))
loss = compute_loss(logits, y.unsqueeze(0))
print(f"Initial loss: {loss.item():.4f}")


# Loss is expected to be log(n_vocab)

sampler = Sampler(model)
context = torch.tensor([enc.encode("Once upon a time")], device=device)
generated = sampler.sample(context, max_length=50)

enc.decode(generated[0].cpu().numpy().tolist())

Initial loss: 11.0485


'Once upon a timembol0 Eas environmentchevthem 625 partially filingSimply admirable 698 acutelypend Rising bottom�ordableisc***** bystandersousing Frag escapes Ethicsospital cramThomas snug covert concerning 88 Nicoleformat 164 Gad consultations margin Euras famed Esther governConsumerDIV allmioken die StewartApparently'

In [ ]:
for step, batch in enumerate(train_loader):
    train_step(model, batch, optimizer, metrics)
    break
    if step % 50 == 0:
        print(f"Step {step}: {metrics.compute()['loss']}")
    if step > 1000:
        break

print(metrics.compute())

/Users/samarth/Workspace/Learning-DL/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
'[Errno 9] Bad file descriptor' thrown while requesting GET https://huggingface.co/datasets/roneneldan/TinyStories/resolve/f54c09fd23315a6f9c86f9dc80f725de7d8f9c64/data/train-00001-of-00004-5852b56a2bd28fd9.parquet
Got disconnected from remote data host. Retrying in 5sec [1/20]
Retrying in 1s [Retry 1/5].


{'loss': tensor(10.7949)}


In [ ]:
metrics_history = {
    'train_loss': [],
    'val_loss': [],
}

for epoch in range(1):
    metrics.reset()
    model.train()
    for step, batch in enumerate(train_loader):
        train_step(model, batch, optimizer, metrics)
        if (step + 1) % 1000 == 0:
            print(f"Training loss: {metrics.compute()['loss']} at step = {step + 1}")
    
    for metric, value in metrics.compute().items():
        metrics_history[f"train_{metric.lower()}"].append(value.item())

    # --- Evaluation ---
    metrics.reset()
    model.eval()
    for batch in val_loader:
        eval_step(model, batch, metrics)
    
    for metric, value in metrics.compute().items():
        metrics_history[f"test_{metric.lower()}"].append(value.item())
    plot_metrics(metrics_history)
    
    latest_acc = metrics_history['test_accuracy'][-1]
    latest_loss = metrics_history['test_loss'][-1]
    print(f"Epoch {epoch+1} complete. Test Acc: {latest_acc:.4f}, Test Loss: {latest_loss:.4f}")

# Attention

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from model import AttentionHead
import torch

# x = torch.randint(1, 10,  (1, 4, 2), dtype=torch.float16)
x = torch.randn((1, 4 ,2))
attn = AttentionHead(CONTEXT_LENGTH, 2, 2)

scores = attn(x)
x, scores

In [ ]:
# self attention
B, T, C = 4, 8, 32

x = torch.randn(B, T, C)
mask = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(mask == 0, float('-inf'))
wei = F.softmax(wei, -1)

out = wei @ x

out.shape

In [ ]:
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)


wei = (q @ k.transpose(-2, -1)) / 4
mask = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(mask == 0, float('-inf'))
wei = F.softmax(wei, -1)
wei[0]

out = wei @ value(x)

out.shape, wei[0]